# IS 362 - Project 2: Cleaning and Analyzing Untidy Data

In this project, I will use Python and pandas to clean and analyze three different untidy datasets from our Week 5 Discussion. The datasets focus on basketball player statistics, public library usage, and online store sales.

My goal is to transform each dataset into a more organized format, perform the proposed analysis, and explain what I learned from the results.

## Dataset 1: Basketball Player Statistics

In [1]:
import pandas as pd

basketball_data = {
    "Player": ["Marcus Reed", "Jordan Ellis", "Devin Carter", "Andre Lewis"],
    "Team": ["New York", "Miami", "Boston", "Chicago"],
    "Game Stats": [
        "24 PTS, 7 REB, 5 AST",
        "18 PTS, 11 REB, 3 AST",
        "29 PTS, 4 REB, 8 AST",
        "21 PTS, 6 REB, 7 AST"
    ],
    "Shooting": ["9/17 FG", "7/14 FG", "11/20 FG", "8/16 FG"]
}

basketball = pd.DataFrame(basketball_data)

basketball.to_csv("basketball_stats.csv", index=False)

basketball

,Player,Team,Game Stats,Shooting
0,Marcus Reed,New York,"24 PTS, 7 REB, 5 AST",9/17 FG
1,Jordan Ellis,Miami,"18 PTS, 11 REB, 3 AST",7/14 FG
2,Devin Carter,Boston,"29 PTS, 4 REB, 8 AST",11/20 FG
3,Andre Lewis,Chicago,"21 PTS, 6 REB, 7 AST",8/16 FG


In [2]:
basketball_raw = pd.read_csv("basketball_stats.csv")

basketball_raw.head()

,Player,Team,Game Stats,Shooting
0,Marcus Reed,New York,"24 PTS, 7 REB, 5 AST",9/17 FG
1,Jordan Ellis,Miami,"18 PTS, 11 REB, 3 AST",7/14 FG
2,Devin Carter,Boston,"29 PTS, 4 REB, 8 AST",11/20 FG
3,Andre Lewis,Chicago,"21 PTS, 6 REB, 7 AST",8/16 FG


In [3]:
basketball_clean = basketball_raw.copy()

basketball_clean[["Points", "Rebounds", "Assists"]] = (
    basketball_clean["Game Stats"]
    .str.extract(r"(\d+) PTS, (\d+) REB, (\d+) AST")
    .astype(int)
)

basketball_clean

,Player,Team,Game Stats,Shooting,Points,Rebounds,Assists
0,Marcus Reed,New York,"24 PTS, 7 REB, 5 AST",9/17 FG,24,7,5
1,Jordan Ellis,Miami,"18 PTS, 11 REB, 3 AST",7/14 FG,18,11,3
2,Devin Carter,Boston,"29 PTS, 4 REB, 8 AST",11/20 FG,29,4,8
3,Andre Lewis,Chicago,"21 PTS, 6 REB, 7 AST",8/16 FG,21,6,7


In [4]:
basketball_clean[["FG_Made", "FG_Attempted"]] = (
    basketball_clean["Shooting"]
    .str.extract(r"(\d+)/(\d+) FG")
    .astype(int)
)

basketball_clean["FG_Percentage"] = (
    basketball_clean["FG_Made"] /
    basketball_clean["FG_Attempted"] * 100
).round(2)

basketball_clean

,Player,Team,Game Stats,Shooting,Points,Rebounds,Assists,FG_Made,FG_Attempted,FG_Percentage
0,Marcus Reed,New York,"24 PTS, 7 REB, 5 AST",9/17 FG,24,7,5,9,17,52.94
1,Jordan Ellis,Miami,"18 PTS, 11 REB, 3 AST",7/14 FG,18,11,3,7,14,50.00
2,Devin Carter,Boston,"29 PTS, 4 REB, 8 AST",11/20 FG,29,4,8,11,20,55.00
3,Andre Lewis,Chicago,"21 PTS, 6 REB, 7 AST",8/16 FG,21,6,7,8,16,50.00


In [5]:
basketball_tidy = basketball_clean.drop(
    columns=["Game Stats", "Shooting"]
)

basketball_tidy

,Player,Team,Points,Rebounds,Assists,FG_Made,FG_Attempted,FG_Percentage
0,Marcus Reed,New York,24,7,5,9,17,52.94
1,Jordan Ellis,Miami,18,11,3,7,14,50.00
2,Devin Carter,Boston,29,4,8,11,20,55.00
3,Andre Lewis,Chicago,21,6,7,8,16,50.00


In [6]:
basketball_ranking = basketball_tidy.sort_values(
    by="Points",
    ascending=False
)

basketball_ranking[
    ["Player", "Team", "Points", "Rebounds", "Assists", "FG_Percentage"]
]

,Player,Team,Points,Rebounds,Assists,FG_Percentage
2,Devin Carter,Boston,29,4,8,55.00
0,Marcus Reed,New York,24,7,5,52.94
3,Andre Lewis,Chicago,21,6,7,50.00
1,Jordan Ellis,Miami,18,11,3,50.00


### Data Cleaning and Transformation

The original basketball dataset was untidy because multiple statistics were stored together in the "Game Stats" and "Shooting" columns. I used pandas to separate points, rebounds, assists, field goals made, and field goals attempted into individual numerical columns.

I also calculated each player's field goal percentage and removed the original combined columns. The cleaned dataset is easier to analyze because each statistic now has its own column.

### Analysis and Conclusions

After organizing the data, I compared the four players based on points, rebounds, assists, and shooting percentage.

Devin Carter had the highest scoring total with 29 points, the most assists with 8, and the highest shooting percentage at 55%. Jordan Ellis led in rebounds with 11, while Marcus Reed scored 24 points and shot 52.94% from the field.

One limitation is that the dataset only includes one set of statistics per player, so I cannot compare performance across multiple seasons as originally proposed in my Week 5 Discussion.

Overall, this analysis showed me how separating combined text values into numerical columns makes it easier to compare player performance using pandas.

## Dataset 2: Public Library Usage

This dataset comes from Joseph Damiano's Week 5 Discussion post. It contains fictional monthly library usage counts for three branches: Downtown, Northside, and West End.

The original dataset separates age groups into different columns and includes two types of services: books borrowed and computer sessions.

I will use pandas to reorganize the data, compare library usage across branches and age groups, and identify patterns in the results.

In [7]:

library_data = {
    "Branch": [
        "Downtown", "Downtown",
        "Northside", "Northside",
        "West End", "West End"
    ],
    "Service Type": [
        "Books Borrowed", "Computer Sessions",
        "Books Borrowed", "Computer Sessions",
        "Books Borrowed", "Computer Sessions"
    ],
    "Kids (0-12)": [520, 110, 340, 85, 210, 45],
    "Teens (13-18)": [180, 320, 115, 210, 90, 160],
    "Adults (19-64)": [940, 580, 620, 390, 430, 290],
    "Seniors (65+)": [610, 210, 450, 140, 380, 95]
}

library = pd.DataFrame(library_data)

library.to_csv("library_usage.csv", index=False)

library


,Branch,Service Type,Kids (0-12),Teens (13-18),Adults (19-64),Seniors (65+)
0,Downtown,Books Borrowed,520,180,940,610
1,Downtown,Computer Sessions,110,320,580,210
2,Northside,Books Borrowed,340,115,620,450
3,Northside,Computer Sessions,85,210,390,140
4,West End,Books Borrowed,210,90,430,380
5,West End,Computer Sessions,45,160,290,95


In [8]:

library_raw = pd.read_csv("library_usage.csv")

library_raw.head()


,Branch,Service Type,Kids (0-12),Teens (13-18),Adults (19-64),Seniors (65+)
0,Downtown,Books Borrowed,520,180,940,610
1,Downtown,Computer Sessions,110,320,580,210
2,Northside,Books Borrowed,340,115,620,450
3,Northside,Computer Sessions,85,210,390,140
4,West End,Books Borrowed,210,90,430,380


In [9]:

library_tidy = pd.melt(
    library_raw,
    id_vars=["Branch", "Service Type"],
    var_name="Age Group",
    value_name="Usage Count"
)

library_tidy


,Branch,Service Type,Age Group,Usage Count
0,Downtown,Books Borrowed,Kids (0-12),520
1,Downtown,Computer Sessions,Kids (0-12),110
2,Northside,Books Borrowed,Kids (0-12),340
3,Northside,Computer Sessions,Kids (0-12),85
4,West End,Books Borrowed,Kids (0-12),210
5,West End,Computer Sessions,Kids (0-12),45
6,Downtown,Books Borrowed,Teens (13-18),180
7,Downtown,Computer Sessions,Teens (13-18),320
8,Northside,Books Borrowed,Teens (13-18),115
9,Northside,Computer Sessions,Teens (13-18),210


In [10]:

library_branch_totals = (
    library_tidy.groupby("Branch")["Usage Count"]
    .sum()
    .sort_values(ascending=False)
)

library_branch_totals


Branch
Downtown     3470
Northside    2350
West End     1700
Name: Usage Count, dtype: int64

In [11]:

library_age_totals = (
    library_tidy.groupby("Age Group")["Usage Count"]
    .sum()
    .sort_values(ascending=False)
)

library_age_totals


Age Group
Adults (19-64)    3250
Seniors (65+)     1885
Kids (0-12)       1310
Teens (13-18)     1075
Name: Usage Count, dtype: int64

In [12]:

library_service_totals = (
    library_tidy.groupby("Service Type")["Usage Count"]
    .sum()
    .sort_values(ascending=False)
)

library_service_totals


Service Type
Books Borrowed       4885
Computer Sessions    2635
Name: Usage Count, dtype: int64


### Data Cleaning and Transformation

The original public library dataset was untidy because the four age groups were stored as separate columns. I first created a CSV file using the information from Joseph Damiano's discussion post and then loaded it into pandas.

I used `pd.melt()` to transform the dataset from a wide format into a tidy format. This created two new columns, "Age Group" and "Usage Count," while keeping the branch and service type information. The original six rows became 24 rows, with each row representing one combination of branch, service type, and age group.

This transformation made it easier to compare library usage using pandas without losing any of the original information.

### Analysis and Conclusions

After cleaning the dataset, I compared library usage by branch, age group, and service type.

Downtown had the highest total usage with 3,470, followed by Northside with 2,350 and West End with 1,700.

Adults between 19 and 64 had the highest usage count at 3,250, followed by seniors with 1,885. Kids had 1,310, while teenagers had the lowest count at 1,075.

I also found that books borrowed accounted for 4,885 uses, compared to 2,635 computer sessions. This suggests that borrowing books was more common than using library computers in this fictional dataset.

Overall, this analysis helped me understand how transforming a wide dataset into a tidy format makes it easier to identify patterns and compare different categories. One limitation is that these are fictional monthly service counts, not actual library visitor records.



## Dataset 3: Online Store Sales

This dataset comes from Nathan Fernandez's Week 5 Discussion post. It contains fictional monthly sales figures for three products: Laptop Stand, Wireless Mouse, and Keyboard.

The original dataset is in a wide format because each month from January through May is stored in a separate column.

I will use pandas to transform the data into a tidy format and analyze monthly sales trends, compare product performance, and identify which products had the highest sales.


In [13]:

sales_data = {
    "Product": ["Laptop Stand", "Wireless Mouse", "Keyboard"],
    "January": [42, 75, 54],
    "February": [51, 81, 61],
    "March": [47, 69, 57],
    "April": [63, 92, 73],
    "May": [58, 88, 70]
}

sales = pd.DataFrame(sales_data)

sales.to_csv("online_store_sales.csv", index=False)

sales


,Product,January,February,March,April,May
0,Laptop Stand,42,51,47,63,58
1,Wireless Mouse,75,81,69,92,88
2,Keyboard,54,61,57,73,70


In [14]:

sales_raw = pd.read_csv("online_store_sales.csv")

sales_raw.head()


,Product,January,February,March,April,May
0,Laptop Stand,42,51,47,63,58
1,Wireless Mouse,75,81,69,92,88
2,Keyboard,54,61,57,73,70


In [15]:

sales_tidy = pd.melt(
    sales_raw,
    id_vars=["Product"],
    var_name="Month",
    value_name="Sales"
)

sales_tidy


,Product,Month,Sales
0,Laptop Stand,January,42
1,Wireless Mouse,January,75
2,Keyboard,January,54
3,Laptop Stand,February,51
4,Wireless Mouse,February,81
5,Keyboard,February,61
6,Laptop Stand,March,47
7,Wireless Mouse,March,69
8,Keyboard,March,57
9,Laptop Stand,April,63


In [16]:

sales_product_totals = (
    sales_tidy.groupby("Product")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

sales_product_totals


Product
Wireless Mouse    405
Keyboard          315
Laptop Stand      261
Name: Sales, dtype: int64

In [17]:

sales_month_totals = (
    sales_tidy.groupby("Month")["Sales"]
    .sum()
    .reindex(["January", "February", "March", "April", "May"])
)

sales_month_totals


Month
January     171
February    193
March       173
April       228
May         216
Name: Sales, dtype: int64

In [18]:

sales_trends = sales_tidy.pivot(
    index="Month",
    columns="Product",
    values="Sales"
)

sales_trends = sales_trends.reindex(
    ["January", "February", "March", "April", "May"]
)

sales_trends


Product,Keyboard,Laptop Stand,Wireless Mouse
Month,,,
January,54,42,75
February,61,51,81
March,57,47,69
April,73,63,92
May,70,58,88


## Data Cleaning and Transformation

The original online store sales dataset was untidy because each month was stored in a separate column. I created a CSV file using the fictional sales data from Nathan Fernandez's Week 5 Discussion post and loaded it into pandas.

I used `pd.melt()` to reorganize the dataset into three columns: Product, Month, and Sales. This changed the original three rows into 15 rows, making it easier to compare sales across different months.

## Analysis and Conclusions

After cleaning the dataset, I compared total sales by product and by month.

Wireless Mouse had the highest total sales with 405, followed by Keyboard with 315 and Laptop Stand with 261.

April had the highest overall sales with 228, while January had the lowest with 171. All three products experienced a decrease in March, followed by an increase in April.

I also noticed that Wireless Mouse had the highest sales every month. This suggests it was the most popular product in this fictional dataset.

Overall, this analysis helped me understand how organizing monthly sales data into a tidy format makes it easier to identify trends and compare product performance. One limitation is that the dataset only covers five months and contains fictional sales figures.